In [2]:
import sqlite3
import requests
import time
import json
import pandas as pd
import urllib.parse

In [3]:
DB_PATH = 'gaming_warehouse.db'

In [4]:
#Crear la tabla de hsitorico de reviews
def preparar_tabla_reviews():
    conn = sqlite3.connect(DB_PATH)
    cursor = conn.cursor()
    cursor.execute("""
        CREATE TABLE IF NOT EXISTS RAW_Steam_Reviews (
            resena_id TEXT PRIMARY KEY,
            juego_id INTEGER,
            resena_texto TEXT,
            recomendado INTEGER, -- (voted_up)
            votos_utiles INTEGER, -- (votes_up)
            votos_graciosos INTEGER, -- (votes_funny)
            puntuacion_ponderada REAL, -- (weighted_vote_score)
            minutos_al_resenar INTEGER, -- (playtime_at_review)
            minutos_totales INTEGER, -- (playtime_forever)
            fecha_creacion_unix INTEGER, -- (timestamp_created)
            autor_num_resenas INTEGER,
            autor_num_juegos INTEGER,
            recibido_gratis INTEGER,
            escrito_acceso_anticipado INTEGER,
            FOREIGN KEY(juego_id) REFERENCES CAT_Juego(juego_id)
        );
    """)
    conn.commit()
    conn.close()


In [5]:
preparar_tabla_reviews()

In [6]:
#Nota: Ver como limitar y tener cuudado con el autocompeltado ya que pone nombres a cosas y me condunde
#Corregir nombre a los esparado
conn = sqlite3.connect(DB_PATH)
cursor = conn.cursor()
cursor.execute("ALTER TABLE RAW_Steam_Reviews RENAME TO Hist_Steam_Reviews;")
conn.commit()
conn.close()

OperationalError: there is already another table or index with this name: Hist_Steam_Reviews

In [7]:
# descargoi las reviews pero estableci apraemtros que la primer version quiso desacrgar todo
# max_reviews_por_juego: Limite máximo de reseñas a descargar por juego.
# min_reviews_juego: Número mínimo de reseñas que debe tener un juego para ser procesado porque hay algunos con pocas reviews.
# limite_juegos: Número máximo de juegos a procesar pro ejecucion.
def descargar_reviews_steam(max_reviews_por_juego=100, min_reviews_juego=20, limite_juegos=10):
    conn = sqlite3.connect(DB_PATH)
    query = """
        SELECT juego_id, id_steam, titulo, recommendations_count 
        FROM CAT_Juego 
        WHERE id_steam IS NOT NULL 
          AND recommendations_count >= ? 
          AND steam_price_final IS NOT NULL
        ORDER BY recommendations_count DESC
        LIMIT ?
    """
    juegos = pd.read_sql_query(query, conn, params=(min_reviews_juego, limite_juegos))
    conn.close()

    if juegos.empty:
        print("No se encontraron juegos con los criterios especificados.")
        return

    total_juegos = len(juegos)
    print(f"Inicio {total_juegos} juegos")

# ciclo para descargar las reseñas de cada juego, con paginacion y control de errores
    for i, (_, juego) in enumerate(juegos.iterrows(), 1):
        appid = juego['id_steam']
        juego_id = juego['juego_id']
        reviews_objetivo = min(max_reviews_por_juego, juego['recommendations_count'])
        reviews_acumuladas = 0
        cursor = '*'
        
        print(f"\n[{i}/{total_juegos}] Procesando: {juego['titulo']} (AppID: {appid})")
        print(f"Objetivo: {reviews_objetivo} reseñas.")

        #paginacion para descargar las rseñas en lotes
        while reviews_acumuladas < max_reviews_por_juego:
            cursor_encoded = urllib.parse.quote(cursor)
            url = f"https://store.steampowered.com/appreviews/{appid}?json=1&filter=all&language=spanish&purchase_type=all&cursor={cursor_encoded}&num_per_page=100"
            
            try:
                res = requests.get(url)
                if res.status_code != 200:
                    print(f"Error HTTP {res.status_code}.")
                    break
                
                data = res.json()
                if not data.get('success') or not data.get('reviews'):
                    print("no hay mas resanas")
                    break

                batch = data['reviews']
                insert_data = []
                
                #Guardar mis datos esperados
                for r in batch:
                    insert_data.append((
                        str(r['recommendationid']),
                        juego_id,
                        r['review'],
                        1 if r['voted_up'] else 0,
                        r['votes_up'],
                        r['votes_funny'],
                        float(r['weighted_vote_score']),
                        r['author']['playtime_at_review'],
                        r['author']['playtime_forever'],
                        r['timestamp_created'],
                        r['author']['num_reviews'],
                        r['author']['num_games_owned'],
                        1 if r['received_for_free'] else 0,
                        1 if r['written_during_early_access'] else 0
                    ))

                conn = sqlite3.connect(DB_PATH)
                db_cursor = conn.cursor()
                db_cursor.executemany(f"""
                    INSERT OR IGNORE INTO Hist_Steam_Reviews 
                    VALUES (?,?,?,?,?,?,?,?,?,?,?,?,?,?)
                """, insert_data)
                
                nuevas = db_cursor.rowcount
                conn.commit()
                conn.close()

                reviews_acumuladas += len(batch)
                
                print(f"  > Lote procesado: {len(batch)} obtenidas | {nuevas} nuevas en BD. (Total: {reviews_acumuladas})")

                cursor = data.get('cursor')
                
                if len(batch) < 100:
                    print("  > Se alcanzó el final de las reseñas disponibles.")
                    break
                
                time.sleep(1)

            except Exception as e:
                print(f"Error inesperado: {e}")
                break

In [ ]:
descargar_reviews_steam(max_reviews_por_juego=2000, min_reviews_juego=20, limite_juegos=6544)

Inicio 5520 juegos

[1/5520] Procesando: Counter-Strike 2 (AppID: 730)
Objetivo: 2000 reseñas.
  > Lote procesado: 100 obtenidas | 0 nuevas en BD. (Total: 100)
  > Lote procesado: 100 obtenidas | 0 nuevas en BD. (Total: 200)
  > Lote procesado: 100 obtenidas | 0 nuevas en BD. (Total: 300)
  > Lote procesado: 100 obtenidas | 0 nuevas en BD. (Total: 400)
  > Lote procesado: 100 obtenidas | 0 nuevas en BD. (Total: 500)
  > Lote procesado: 100 obtenidas | 39 nuevas en BD. (Total: 600)
  > Lote procesado: 100 obtenidas | 36 nuevas en BD. (Total: 700)
  > Lote procesado: 100 obtenidas | 47 nuevas en BD. (Total: 800)
  > Lote procesado: 100 obtenidas | 92 nuevas en BD. (Total: 900)
  > Lote procesado: 100 obtenidas | 100 nuevas en BD. (Total: 1000)
  > Lote procesado: 100 obtenidas | 100 nuevas en BD. (Total: 1100)
  > Lote procesado: 100 obtenidas | 100 nuevas en BD. (Total: 1200)
  > Lote procesado: 100 obtenidas | 100 nuevas en BD. (Total: 1300)
  > Lote procesado: 100 obtenidas | 100 nuev

In [ ]:
def resumen_final_reviews(cantidad_top=10):
    conn = sqlite3.connect(DB_PATH)
    query = """
    SELECT 
        j.titulo,
        COUNT(r.resena_id) AS total_resenas_descargadas,
        ROUND(AVG(r.recomendado) * 100, 2) || '%' AS recomendado,
        ROUND(AVG(r.minutos_totales / 60.0), 1) AS promedio_horas_jugadas
    FROM Hist_Steam_Reviews r
    JOIN CAT_Juego j ON r.juego_id = j.juego_id
    GROUP BY j.juego_id
    ORDER BY total_resenas_descargadas DESC
    LIMIT ?;
    """
    df = pd.read_sql_query(query, conn, params=(cantidad_top,))
    conn.close()
    
    display(df)


In [ ]:
resumen_final_reviews(50)


,titulo,total_resenas_descargadas,recomendado,promedio_horas_jugadas
0,Terraria,503,96.62%,146.9
1,Counter-Strike 2,501,69.46%,583.9
2,Garry's Mod,501,98.4%,78.8
3,EA Sports FC 26,500,50.0%,86.4
4,Mewgenics,500,97.2%,88.2
5,Crimson Desert,500,99.6%,59.8
6,Schedule I,500,96.4%,43.7
7,R.E.P.O.,500,95.4%,43.5
8,Resident Evil Requiem,500,99.8%,37.4
9,Project Zomboid,500,95.4%,197.0


In [ ]:
def inspeccionar_resenas(busqueda, es_id=False):
    conn = sqlite3.connect(DB_PATH)
    
    if es_id:
        filtro = "j.juego_id = ?"
        parametro = busqueda
    else:
        filtro = "j.titulo LIKE ?"
        parametro = f"%{busqueda}%"

    query = f"""
    SELECT 
        j.juego_id,
        j.titulo,
        r.recomendado,
        r.minutos_totales / 60 AS horas_jugadas,
        r.votos_utiles,
        r.resena_texto
    FROM Hist_Steam_Reviews r
    JOIN CAT_Juego j ON r.juego_id = j.juego_id
    WHERE {filtro}
    --ORDER BY r.votos_utiles DESC
    --LIMIT 10;
    """
    
    df = pd.read_sql_query(query, conn, params=(parametro,))
    conn.close()
    
    if df.empty:
        print(f"No se encontraron reseñas para: {busqueda}")
    else:
        display(df)

In [ ]:
#Ara que tratar de alguna forma las malas palabras de los jeugos ya que se ve que son creativos con las opiniones
inspeccionar_resenas("lego")


,juego_id,titulo,recomendado,horas_jugadas,votos_utiles,resena_texto
0,1095,LEGO Star Wars: The Skywalker Saga,1,92,9,"92 horas después y con el 100% completado, pue..."
1,1095,LEGO Star Wars: The Skywalker Saga,1,11,0,"Es un gran juego, buenos graficos, dinamicas y..."
2,1095,LEGO Star Wars: The Skywalker Saga,1,32,0,"Esta muy bien hecha la trilogia original, much..."
3,1095,LEGO Star Wars: The Skywalker Saga,1,55,0,"es muy diferente a los originales de play 2 , ..."
4,1095,LEGO Star Wars: The Skywalker Saga,1,5,0,siempre los lego estan buenos pero prefiero ma...
...,...,...,...,...,...,...
261,1564,LEGO Indiana Jones 2: The Adventure Continues,1,18,0,"ESTA BIEN BUENO, LO PUTISIMO MALO ES QUE LA MI..."
262,1564,LEGO Indiana Jones 2: The Adventure Continues,1,17,0,clasicazo
263,3902,LEGO Harry Potter Collection,1,57,0,Obra maestra
264,4659,LEGO 2K Drive,1,26,0,"Esta muy guapo, el mundo es bastante grande y ..."


In [23]:
inspeccionar_resenas("pubg")

--- Mostrando reseñas más útiles para: PUBG: Battlegrounds ---


,juego_id,titulo,recomendado,horas_jugadas,votos_utiles,resena_texto
0,226,PUBG: Battlegrounds,1,320,0,"Excelente juego, mundo abierto y realista con ..."
1,226,PUBG: Battlegrounds,1,1087,0,"Me gusta el juego, siempre estan cambiando cos..."
2,226,PUBG: Battlegrounds,0,300,3,"Paso de jugar, es imposible con la cantidad de..."
3,226,PUBG: Battlegrounds,0,68,1,Imagina ser un muerto de hambre y sacar a la c...
4,226,PUBG: Battlegrounds,1,682,2,Gracias por poner la posibilidad de meter a un...
5,226,PUBG: Battlegrounds,1,2230,0,"Juego realista, mecanicas realistas, competiti..."
6,226,PUBG: Battlegrounds,1,687,0,este juego le dedique gran parte de mi vida ga...
7,226,PUBG: Battlegrounds,0,110,0,"Hace tiempo que tira error de ""Crash Unknow"" -..."
8,226,PUBG: Battlegrounds,1,710,0,ete juego esta chevere lo juego con mi pana de...
9,226,PUBG: Battlegrounds,0,5281,0,mierda de juego mal hecho cada ves que le mete...
